## Load Libraries

In [1]:
import pandas as pd
import numpy as np

## Helper Functions

In [2]:
column_translation_weather = {
    "ID OMM station": "WMO_Station_ID",
    "Date": "Date",
    "Pression au niveau mer": "Sea_Level_Pressure_hPa",
    "Variation de pression en 3 heures": "Pressure_Change_3h_hPa",
    "Type de tendance barométrique": "Barometric_Trend_Type",
    "Direction du vent moyen 10 mn": "Wind_Direction_10min_deg",
    "Vitesse du vent moyen 10 mn": "Wind_Speed_10min_mps",
    "Température": "Temperature_K",
    "Température (°C)": "Temperature_C",
    "Point de rosée": "Dew_Point_C",
    "Humidité": "Relative_Humidity_percent",
    "Visibilité horizontale": "Horizontal_Visibility_m",
    "Temps présent": "Current_Weather",
    "Temps passé 1": "Past_Weather_1",
    "Temps passé 2": "Past_Weather_2",
    "Nebulosité totale": "Total_Cloud_Cover_okta",
    "Nébulosité des nuages de l' étage inférieur": "Low_Level_Cloud_Cover_okta",
    "Hauteur de la base des nuages de l'étage inférieur": "Low_Cloud_Base_Height_m",
    "Type des nuages de l'étage inférieur": "Low_Cloud_Type",
    "Type des nuages de l'étage moyen": "Mid_Cloud_Type",
    "Type des nuages de l'étage supérieur": "High_Cloud_Type",
    "Pression station": "Station_Pressure_hPa",
    "Niveau barométrique": "Barometric_Level",
    "Géopotentiel": "Geopotential_m2s2",
    "Variation de pression en 24 heures": "Pressure_Change_24h_hPa",
    "Température minimale sur 12 heures": "Min_Temperature_12h_C",
    "Température minimale sur 24 heures": "Min_Temperature_24h_C",
    "Température maximale sur 12 heures": "Max_Temperature_12h_C",
    "Température maximale sur 24 heures": "Max_Temperature_24h_C",
    "Température minimale du sol sur 12 heures": "Min_Ground_Temperature_12h_C",
    "Méthode de mesure Température du thermomètre mouillé": "Wet_Bulb_Measurement_Method",
    "Température du thermomètre mouillé": "Wet_Bulb_Temperature_C",
    "Rafale sur les 10 dernières minutes": "Wind_Gust_Last10min_mps",
    "Rafales sur une période": "Wind_Gust_Period_mps",
    "Periode de mesure de la rafale": "Gust_Measurement_Period",
    "Etat du sol": "Ground_State",
    "Hauteur totale de la couche de neige, glace, autre au sol": "Total_Snow_Ice_Depth_cm",
    "Hauteur de la neige fraîche": "Fresh_Snow_Depth_cm",
    "Periode de mesure de la neige fraiche": "Fresh_Snow_Measurement_Period",
    "Précipitations dans la dernière heure": "Precipitation_Last1h_mm",
    "Précipitations dans les 3 dernières heures": "Precipitation_Last3h_mm",
    "Précipitations dans les 6 dernières heures": "Precipitation_Last6h_mm",
    "Précipitations dans les 12 dernières heures": "Precipitation_Last12h_mm",
    "Précipitations dans les 24 dernières heures": "Precipitation_Last24h_mm",
    "Phénomène spécial 1": "Special_Weather_Event_1",
    "Phénomène spécial 2": "Special_Weather_Event_2",
    "Phénomène spécial 3": "Special_Weather_Event_3",
    "Phénomène spécial 4": "Special_Weather_Event_4",
    "Nébulosité couche nuageuse 1": "Cloud_Layer1_Cover_okta",
    "Type nuage 1": "Cloud_Layer1_Type",
    "Hauteur de base 1": "Cloud_Layer1_Base_Height_m",
    "Nébulosité couche nuageuse 2": "Cloud_Layer2_Cover_okta",
    "Type nuage 2": "Cloud_Layer2_Type",
    "Hauteur de base 2": "Cloud_Layer2_Base_Height_m",
    "Nébulosité couche nuageuse 3": "Cloud_Layer3_Cover_okta",
    "Type nuage 3": "Cloud_Layer3_Type",
    "Hauteur de base 3": "Cloud_Layer3_Base_Height_m",
    "Nébulosité couche nuageuse 4": "Cloud_Layer4_Cover_okta",
    "Type nuage 4": "Cloud_Layer4_Type",
    "Hauteur de base 4": "Cloud_Layer4_Base_Height_m",
    "Nom": "Station_Name",
    "Coordonnees": "Coordinates",
    "Latitude": "Latitude",
    "Longitude": "Longitude",
    "Altitude": "Altitude_m",
    "communes (name)": "Municipality_Name",
    "communes (code)": "Municipality_Code",
    "EPCI (name)": "Intermunicipal_Group_Name",
    "EPCI (code)": "Intermunicipal_Group_Code",
    "department (name)": "Department_Name",
    "department (code)": "Department_Code",
    "region (name)": "Region_Name",
    "region (code)": "Region_Code",
    "mois_de_l_annee": "Month_of_Year"
}

import pandas as pd
import numpy as np

def load_and_clean_weather(file_path):

    # Load dataset
    df = pd.read_excel(file_path)

    # Clean column names
    df.columns = df.columns.str.strip()

    # Drop duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Rename columns to English
    df.rename(columns=column_translation_weather, inplace=True)

    # Replace missing values coded as ND
    df.replace("ND", np.nan, inplace=True)

    # Convert Date column to datetime
    df["Datetime"] = pd.to_datetime(
        df["Date"],
        dayfirst=True,
        errors="coerce",
        utc=True
    ).dt.tz_localize(None)

    # Convert numeric columns
    df["Temperature_C"] = pd.to_numeric(df["Temperature_C"], errors="coerce") - 273.15
    df["Altitude_m"] = pd.to_numeric(df["Altitude_m"], errors="coerce")

    # Keep only relevant variables
    df = df[[
        "Datetime",
        "WMO_Station_ID",
        "Temperature_C",
        "Altitude_m"
    ]]

    # Remove rows with missing key variables
    df = df.dropna(subset=["Datetime", "Temperature_C"])

    # Sort by time
    df = df.sort_values("Datetime").reset_index(drop=True)

    # ---- TIME FEATURES ----
    df["Hour"] = df["Datetime"].dt.hour
    df["DayOfWeek"] = df["Datetime"].dt.dayofweek
    df["Month"] = df["Datetime"].dt.month
    df["DayOfYear"] = df["Datetime"].dt.dayofyear
    df["Year"] = df["Datetime"].dt.year

    # Weekend indicator
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

    return df


## Read all csv files and return a combined DataFrame

In [3]:
import glob

# Get all Excel files inside the folder
files = glob.glob("weather/*.xlsx")

# Process each file
dfs = [load_and_clean_weather(file) for file in files]

# Combine all dataframes
df_final = pd.concat(dfs, ignore_index=True)

# Save cleaned dataset
df_final.to_csv("weather/France_Weather_Cleaned.csv", index=False)


/var/folders/4d/2qwqvq397g31zlw8wxxy_0980000gn/T/ipykernel_74337/1768064855.py:99: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Datetime"] = pd.to_datetime(
/var/folders/4d/2qwqvq397g31zlw8wxxy_0980000gn/T/ipykernel_74337/1768064855.py:99: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Datetime"] = pd.to_datetime(
/var/folders/4d/2qwqvq397g31zlw8wxxy_0980000gn/T/ipykernel_74337/1768064855.py:99: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Datetime"] = pd.to_datetime(
/var/folders/4d/2qwqvq397g31zlw8wxxy_0980000gn/T/ipykernel_74337/1768064855.py:99: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified